# Week 4 · IBM Q Experience, Composer, and OpenQASM

**MIT xPRO QCFx1 · Week 4 · Video Q1M4V5**

This notebook covers the QASM material, the connectivity/CNOT-direction issue,
and the "Prediction to settle after the binge" from the end of
`week-4-watch-guide.md`.

Time reference: **Paramaribo (UTC−3)**. Hard deadline: **Mon 2026-11-02 20:30**.

## 1. What the video shows

Since 2016 the IBM Q Experience has offered cloud access to real devices. The
device page shows a chip schematic, a **connectivity graph** (which two-qubit
gates are allowed), per-qubit and per-gate noise numbers, and the dilution fridge
at roughly 21 mK.

The Bell experiment in the video is:

1. $H$ on `q[0]`.
2. `CX q[0] -> q[1]`.
3. Measure both qubits.

On the simulator, the result is about 50% `00` and 50% `11` with the default
~1000 shots. On hardware, noise adds other outcomes.

QASM is a **hardware-agnostic** circuit description: data dependencies, no
explicit timings. Composer generates QASM under the hood. A QASM 2 program has
this shape:

```qasm
OPENQASM 2.0;
include "qelib1.inc";
qreg q[5];
creg c[5];
h q[0];
cx q[0], q[1];
measure q -> c;
```

In [ ]:
from pathlib import Path
import sys

def find_first_code_dir() -> Path:
    candidates = [Path.cwd(), Path.cwd() / "first-code", Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "firstcode_helpers.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repo root, first-code/, or first-code/notebooks/."
    )

FIRSTCODE_DIR = find_first_code_dir()
if str(FIRSTCODE_DIR) not in sys.path:
    sys.path.insert(0, str(FIRSTCODE_DIR))

print(f"First-code folder: {FIRSTCODE_DIR}")

import math
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit
import qiskit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_histogram
import firstcode_helpers as w4

print(f"Qiskit version: {qiskit.__version__}")

In [ ]:
from qiskit import qasm2, qasm3

qasm_source = r'''
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0], q[1];
measure q -> c;
'''

bell_qasm = qasm2.loads(qasm_source)
print(bell_qasm.draw(output="text"))

counts = w4.run_counts(bell_qasm, shots=1024, seed=42)
print("Counts:", dict(sorted(counts.items())))
display(plot_histogram(counts, title="Bell circuit loaded from OpenQASM 2.0"))

### Round-trip the circuit

`qasm2.dumps()` writes OpenQASM 2.0; `qasm3.dumps()` writes OpenQASM 3. The
modern Qiskit parser accepts OpenQASM 3 with `--qasm3` in the checker.

In [ ]:
print("=== OpenQASM 2.0 dump ===")
print(qasm2.dumps(bell_qasm))
print("=== OpenQASM 3.0 dump ===")
print(qasm3.dumps(bell_qasm))

## 2. CNOT direction and the Hadamard sandwich

On the chip in the video, `CX q[2] -> q[3]` was not allowed; only the reverse
direction was native. One way to reverse a CNOT is to surround it with Hadamards
on both qubits:

$$
(H \otimes H)\,\mathrm{CX}_{0\to1}\,(H \otimes H)
= \mathrm{CX}_{1\to0}.
$$

The cost is four extra single-qubit gates, which means more error. The watch
guide asks you to predict whether the two operators are equal before running the
check. Predict first, then run the next cell.

In [ ]:
direct, flipped = w4.cnot_direction_pair()
print("Direct CNOT q1 -> q0:")
print(direct.draw(output="text"))
print("H-sandwich CNOT:")
print(flipped.draw(output="text"))
print("Equivalent up to global phase?", Operator(direct).equiv(Operator(flipped)))

The result is `True`. The H sandwich implements a CNOT with control and target
swapped. Today's Qiskit transpiler will often flip and route CNOTs automatically
for a chosen backend, so the manual sandwich is mainly a way to understand what
the old device page was forcing the user to do.

## 3. Connectivity and transpilation today

A coupling map describes which qubit pairs can interact. If the circuit needs a
CNOT between non-adjacent qubits, the transpiler inserts swaps (or equivalent
routing). The exact output depends on the backend, basis gates, and transpiler
seed.

In [ ]:
from qiskit import transpile
from qiskit.transpiler import CouplingMap

circuit = QuantumCircuit(3)
circuit.cx(0, 2)
print("Original circuit:")
print(circuit.draw(output="text"))

coupling = CouplingMap([[0, 1], [1, 2]])
routed = transpile(
    circuit,
    coupling_map=coupling,
    basis_gates=["cx", "h", "rz", "sx", "x"],
    seed_transpiler=42,
)
print("Routed for 0-1-2 connectivity:")
print(routed.draw(output="text"))
print("Operations after routing:", dict(routed.count_ops()))

## 4. Noise and the real-device option

The local cells use Aer or `StatevectorSampler`, so they work offline. Connecting
to IBM Quantum is optional and requires a token in the environment. The token
must never be hard-coded or committed.

```bash
export QISKIT_IBM_TOKEN="..."
```

The next cell only tries to connect when that variable is set. It also requires
`qiskit-ibm-runtime`, which is not part of the minimal Week 4 environment.

In [ ]:
import os

token = os.environ.get("QISKIT_IBM_TOKEN")
if not token:
    print("QISKIT_IBM_TOKEN is not set. Local simulation remains available.")
else:
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService
    except ImportError:
        print("qiskit-ibm-runtime is not installed; install it separately to run on hardware.")
    else:
        service = QiskitRuntimeService(channel="ibm_quantum", token=token)
        print("Connected to IBM Quantum. Backends visible:", len(service.backends()))

## 5. 2018 Composer vs today

The video is from about 2018. Today's Composer UI is different, and the
transpiler routes and flips CNOTs automatically. Two rules follow:

- For course questions, answer from the course framing and the video's device.
- For the lab, expect a different UI and let the tooling handle routing.

Do not treat the video's specific qubit numbers or connectivity as today's
device layout.

## 6. Week 4 checklist (Paramaribo, UTC−3)

| Milestone | Paramaribo | UTC |
|---|---|---|
| Week 4 opens | Sun 2026-10-25 21:00 | Mon 2026-10-26 00:00 |
| Videos + in-video questions done | Mon 2026-10-26, end of day | |
| QASM + Qiskit practice, DJ lab, real-device run | Tue 2026-10-27 to Sat 2026-10-31 | |
| Target: every graded item submitted | Sun 2026-11-01 | |
| **Hard deadline: course ends, forums lock** | **Mon 2026-11-02 20:30** | **Mon 2026-11-02 23:30** |

Use **Save**, not **Submit**, until the section's videos and readings are done.
There is one attempt per question. November 2 is buffer, not planning time.

## 7. Predict before you run

1. Do the direct and H-sandwich CNOTs give the same operator? Write your prediction
   before the equivalence cell.
2. Why does the transpiler need extra gates for a non-adjacent CNOT?
3. What is the difference between a hardware-agnostic QASM program and a
   hardware-specific transpiled circuit?
4. Why does the old device page force a direction on a CNOT that is physically
   symmetric in the abstract circuit model?

## 8. qasm-check

The repo includes a checker for hand-written QASM files. From the repository
root:

```bash
uv run --project week-4 python week-4/qasm_check.py week-4/practice/katas/bell.qasm \
  --expect "00:0.5,11:0.5"
```

It prints the Qiskit version, parses the file, draws the circuit if asked,
simulates it locally, and reports PASS/FAIL per expected bitstring. It is a
checker, not a circuit generator.